In [1]:
import cv2 as cv
import os
import matplotlib as plt
import random

Functia de extragere a fetelor

In [14]:
def extract_faces():
    folder_names = ["daphne", "fred", "shaggy", "velma"]


    if not os.path.exists("faces_for72_2"):
        os.makedirs("faces_for72_2")
        
    
    for folder in folder_names:
        file_path = "antrenare/" + folder + "_annotations.txt"
        annotations_file = open(file_path, "r")
        file_content = annotations_file.readlines()
        index = 0
        
        current_folder = "antrenare/" + folder
        for img in os.listdir(current_folder):
            image_path = os.path.join(current_folder, img)
            image = cv.imread(image_path)
            image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
            
            line = file_content[index].split()
        
            while index < len(file_content) and file_content[index].split()[0] == img:
                line = file_content[index].split()
                
                pad = 0

                ymin = max(0, int(line[2]) - pad)
                ymax = min(image.shape[0], int(line[4]) + pad)
                xmin = max(0, int(line[1]) - pad)
                xmax = min(image.shape[1], int(line[3]) + pad)

                if xmax-xmin >= 50 and ymax-ymin >= 50:
                
                    cropped_image = image[ymin:ymax, xmin:xmax]
                    
                    filename = f"face_{folder}_{index}.jpg"
                    path_out = os.path.join("faces_for72_2", filename)
                    cv.imwrite(path_out, cropped_image)
                    
                index += 1
            
    

In [13]:
extract_faces()

Functia de redimensionare pentru fete

In [15]:
def resize_template(input_folder, output_folder, size = (72, 72)):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    id = 0
    for filename in os.listdir(input_folder):
        image_path = os.path.join(input_folder, filename)
        resize_path = os.path.join(output_folder, filename)

        img = cv.imread(image_path)
        img_resized = cv.resize(img, size, interpolation=cv.INTER_AREA)
        filename = f"72px_{id}.jpg"
        cv.imwrite(os.path.join(output_folder, filename), img_resized)
        
        id += 1
        
    print("\nCompleted")

In [16]:
input_folder = "faces_for72_2"
output_folder = "faces_72x72_2"

resize_template(input_folder, output_folder)


Completed


Functie de extragere a negativelor random

In [17]:
def negative_examples_extract(rand_per_image = 2, size = (72, 72)):
    folder_names = ["daphne", "fred", "shaggy", "velma"]

    #146789
    random.seed(986435)

    if not os.path.exists("negative_examples_72x72"):
        os.makedirs("negative_examples_72x72")
    
    negative_index = 0
    
    for folder in folder_names:
        file_path = "antrenare/" + folder + "_annotations.txt"
        annotations_file = open(file_path, "r")
        file_content = annotations_file.readlines()
        index = 0
        
        current_folder = "antrenare/" + folder
        for img in os.listdir(current_folder):
            image_path = os.path.join(current_folder, img)
            image = cv.imread(image_path)
            image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
            
            
            line = file_content[index].split()
        
            face_coordinates = []
            while index < len(file_content) and file_content[index].split()[0] == img:
                line = file_content[index].split()
                
                ymin = int(line[2])
                ymax = int(line[4])
                xmin = int(line[1])
                xmax = int(line[3])
                face_coordinates.append([xmin, ymin, xmax, ymax])
                
                index += 1
                
            
            id = 0
            
            while id < rand_per_image:
                x = random.randint(0, 480 - size[0])
                y = random.randint(0, 360 - size[1])
                
                overlap = False
                
                for xmin, ymin, xmax, ymax in face_coordinates:
                        if not (x + size[0] <= xmin  or x >= xmax or y + size[1] <= ymin or y >= ymax ):
                            overlap = True
                            break
            
                if overlap == False:
                    y_max = y + size[1]
                    x_max = x + size[0]
                    
                    cropped_image = image[y: y_max, x: x_max]
                    
                    filename = f"random_{negative_index}.jpg"
                    cv.imwrite(os.path.join("negative_examples_72x72", filename), cropped_image)
                    
                    id +=1 
                    negative_index += 1
                    
             

In [18]:
negative_examples_extract()

Informatii despre imagini

In [2]:
def faces_min_max():
    widths = []
    heights = []
    files = []

    for img_name in os.listdir("faces_cropped_4px"):
        image_path = os.path.join("faces_cropped_4px", img_name)
        image = cv.imread(image_path)
        
        h, w = image.shape[:2]
        widths.append(w)
        heights.append(h)
        files.append(img_name)

    min_height = min(heights)
    max_height = max(heights)
    min_width = min(widths)
    max_width = max(widths)
    avg_height = sum(heights) / len(heights)
    avg_width = sum(widths) / len(widths)

    min_idx = heights.index(min_height)
    max_idx = heights.index(max_height)

    print(f"Cea mai mica imagine: {files[min_idx]} cu {min_width}x{min_height}")
    print(f"Cea mai mare imagine: {files[max_idx]} cu {max_width}x{max_height}")
    print(f"Dimensiune medie: {avg_width:.2f}x{avg_height:.2f}")


In [3]:
faces_min_max()

Cea mai mica imagine: pad_velma_162.jpg cu 14x9
Cea mai mare imagine: pad_velma_732.jpg cu 185x284
Dimensiune medie: 70.77x87.80
